In [1]:
%%capture
!pip install -U crawl4ai
!pip install nest_asyncio

In [2]:
import crawl4ai
print(crawl4ai.__version__.__version__)

0.4.248


In [3]:
%%capture
!crawl4ai-setup

In [4]:
import requests
import os
import re
from bs4 import BeautifulSoup
from tqdm import tqdm
from playwright.async_api import async_playwright

In [7]:
class VbplCrawler:
    def __init__(self, root_path="BoPhapDienDienTu"):
        self.base_url = "http://vbpl.vn/TW/Pages/vbpq"
        self.root = root_path
        self.item_ids = self.get_ids()

        self.output_dirs = {
            "vbpl": f"{self.root}/vbpl",
            "history": f"{self.root}/history",
            "related": f"{self.root}/related",
            "property": f"{self.root}/property",
            "pdf": f"{self.root}/pdf"
        }

        # Ensure directories exist
        for path in self.output_dirs.values():
            os.makedirs(path, exist_ok=True)

        self.url_mappings = {
            "vbpl": "toanvan",
            "history": "lichsu",
            "related": "vanbanlienquan",
            "property": "thuoctinh",
            "pdf": "van-ban-goc"
        }

        self.name_mappings = {
            "vbpl": "full",
            "history": "h",
            "related": "r",
            "property": "p",
            "pdf": "pdf"
        }
    def get_ids(self):
        """
        Get unique Item IDs across all index pages
        """
        item_ids = set()
        demuc_path = self.root + "/demuc"

        for file in tqdm(os.listdir(demuc_path)):
            if "html" in file:
              file_path = os.path.join(demuc_path, file)

              with open(file_path, 'r') as f:
                  soup = BeautifulSoup(f, "html.parser")
                  tags = soup.find_all('a', href=True)
                  if tags:
                      for tag in tags:
                          url = tag['href']
                          if url.startswith(self.base_url):
                              try:
                                  item_id = re.search(r'ItemID=(\d+)', url).group(1) # use regex to capture special cases such as ...?ItemID=139689&Keyword=41/2019/TT-BCT.html
                                  item_ids.add(item_id)
                              except:
                                  continue

        return item_ids

    def len(self):
        return len(self.item_ids)

    def _get_and_save_html(self, session: requests.Session, url: str, output_dir:str):
        """
        Get content of url with a requests session and write to output directory
        """
        with open(f"{output_dir}", 'wb') as f:
            with session.get(url) as resp:
                if resp.status_code == 200: # successful requests
                    content = resp.content
                    f.write(content)

    def crawl_vbpl_html(self):
        """
        Crawl the HTML of toanvan, property, history, related articles and the pdf page
        """

        with requests.Session() as s:
            for item_id in tqdm(self.item_ids):
                for key, replacement in self.url_mappings.items():
                    # Example url: https://vbpl.vn/TW/Pages/vbpq-toanvan.aspx?ItemID=148955
                    url = f"{self.base_url}-{self.url_mappings[key]}.aspx?ItemID={item_id}"
                    new_url = url.replace("toanvan", replacement)
                    filepath = f"{self.output_dirs[key]}/{self.name_mappings[key]}_{item_id}.html"
                    self._get_and_save_html(s, new_url, filepath)

    def crawl_vbpl_text(self):
        """
        Optinal method to crawl text of full documents (instead of the full HTML)
        """
        with requests.Session() as s:
            for item_id in self.item_ids:
                full_url = f"{self.base_url}-toanvan.aspx?ItemID={item_id}"
                resp = s.get(full_url)
                if resp.status_code == 200:
                    soup = BeautifulSoup(resp.text, "html.parser")
                    content_div = soup.find("div", class_="fulltext")

                    if content_div:
                        content = content_div.get_text(separator="\n", strip=True)
                        with open(f'{self.root}/vbpl/text/text_{item_id}.txt', 'w', encoding='utf-8') as f:
                            f.write(content)


    # Run seperately in a notebook
    async def crawl_pdf(self):
        """
        Use Crawl4AI to get the pdf because somehow selenium and requests cant get the full HTML with the link even with the headers included
        """
        async with async_playwright() as p:
            browser = await p.chromium.launch(headless=True)
            page = await browser.new_page()

            for item_id in tqdm(self.item_ids):
                await page.goto(f'{self.base_url}-van-ban-goc.aspx?ItemID={item_id}')
                content = await page.content()

                try:
                    # example template: <!--<embed src="https://drive.google.com/viewerng/viewer?embedded=true&url=http://vbpl.vn/FileData/TW/Lists/vbpq/Attachments/167957/VanBanGoc_1_Thong tu 05-TT-BTP ngay 10-6-2024.PDF" width="760" height="995">-->
                    match = re.search(r'src="[^"]+url=([^"]+\.PDF)"', content)
                except:
                    continue

                if match:
                  pdf_link = match.group(1)
                  with requests.Session() as s:
                      # save the pdf
                      output_dir = f"{self.output_dirs['pdf']}/{self.name_mappings['pdf']}_{item_id}.pdf"
                      self._get_and_save_html(s, pdf_link, output_dir)


In [8]:
crawler = VbplCrawler(root_path="/content")

100%|██████████| 294/294 [01:12<00:00,  4.06it/s]


In [9]:
import asyncio

await crawler.crawl_pdf()

100%|██████████| 5943/5943 [1:04:52<00:00,  1.53it/s]


In [10]:
!zip -r /content/pdf.zip /content/pdf

  adding: content/pdf/ (stored 0%)
  adding: content/pdf/pdf_125702.pdf (deflated 0%)
  adding: content/pdf/pdf_37636.pdf (deflated 7%)
  adding: content/pdf/pdf_105676.pdf (deflated 1%)
  adding: content/pdf/pdf_106909.pdf (deflated 11%)
  adding: content/pdf/pdf_159006.pdf (stored 0%)
  adding: content/pdf/pdf_37531.pdf (deflated 16%)
  adding: content/pdf/pdf_24060.pdf (deflated 6%)
  adding: content/pdf/pdf_45502.pdf (deflated 8%)
  adding: content/pdf/pdf_113070.pdf (deflated 7%)
  adding: content/pdf/pdf_105679.pdf (deflated 1%)
  adding: content/pdf/pdf_153024.pdf (deflated 0%)
  adding: content/pdf/pdf_125504.pdf (deflated 0%)
  adding: content/pdf/pdf_122564.pdf (deflated 0%)
  adding: content/pdf/pdf_148467.pdf (deflated 18%)
  adding: content/pdf/pdf_154772.pdf (deflated 0%)
  adding: content/pdf/pdf_138935.pdf (deflated 0%)
  adding: content/pdf/pdf_33649.pdf (deflated 9%)
  adding: content/pdf/pdf_113407.pdf (deflated 9%)
  adding: content/pdf/pdf_150325.pdf (deflated 1%)


In [11]:
from google.colab import files
files.download("/content/pdf.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>